In [3]:
# install dependencies
! pip3 install numpy

You should consider upgrading via the '/Users/wahid/Desktop/projects/small-llm/.venv/bin/python3 -m pip install --upgrade pip' command.


In [1]:
# import necessary module
import numpy as np
from urllib.request import urlopen
from io import BytesIO
import os
import math

In [17]:
def load_model():
    model_path = './models/tiny_english_gpt.npz'
    if os.path.exists(model_path):
        return np.load(model_path, allow_pickle=True)
    
    url = f'https://algo.monster/{model_path}'
    with urlopen(url) as response:
        data = response.read()
        
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        with open(model_path, 'wb') as f:
            f.write(data)

        return np.load(model_path, allow_pickle=True)

def softmax(x):
    # Subtracting np.max prevents numerical overflow with large numbers
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

def gelu(x):
    """GELU: smooth non-linearity"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

In [18]:
model = load_model()
print([k for k in model.keys()])

# load the weights
token_embed_weight = model['token_embed.weight']
pos_embed_weight = model['pos_embed.weight']
w_q_0_w = model['blocks.0.attn.W_q.weight']
w_k_0_w = model['blocks.0.attn.W_k.weight']
w_v_0_w = model['blocks.0.attn.W_v.weight']

w_q_0_b = model['blocks.0.attn.W_q.bias']
w_k_0_b = model['blocks.0.attn.W_k.bias']
w_v_0_b = model['blocks.0.attn.W_v.bias']

ff_line_1_w = model['blocks.0.ff.linear1.weight']
ff_line_2_w = model['blocks.0.ff.linear2.weight']

ff_line_1_b = model['blocks.0.ff.linear1.bias']
ff_line_2_b = model['blocks.0.ff.linear2.bias']

norm_1_w = model['blocks.0.norm1.weight']
norm_2_w = model['blocks.0.norm2.weight']

norm_1_b = model['blocks.0.norm1.bias']
norm_2_b = model['blocks.0.norm2.bias']

lm_head_w = model['lm_head.weight']

vocab = model['vocab'].tolist()

['token_embed.weight', 'pos_embed.weight', 'blocks.0.attn.W_q.weight', 'blocks.0.attn.W_q.bias', 'blocks.0.attn.W_k.weight', 'blocks.0.attn.W_k.bias', 'blocks.0.attn.W_v.weight', 'blocks.0.attn.W_v.bias', 'blocks.0.attn.W_o.weight', 'blocks.0.attn.W_o.bias', 'blocks.0.ff.linear1.weight', 'blocks.0.ff.linear1.bias', 'blocks.0.ff.linear2.weight', 'blocks.0.ff.linear2.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'ln_f.weight', 'ln_f.bias', 'lm_head.weight', 'vocab', 'd_model', 'n_heads', 'n_layers', 'd_ff', 'max_seq_len']


In [4]:
inp_words = ['the', 'cat', 'sat']

In [50]:
def process(words: list[str]):
    if len(words) > 10: return words
    inp_embed = [token_embed_weight[vocab.index(word)] for word in words]

    # add position embedding
    inp_pos_embed = pos_embed_weight[:len(inp_embed)]

    # convert it into numpy array
    inp_array = np.array(inp_pos_embed)

    Q = inp_array @ w_q_0_w + w_q_0_b
    K = inp_array @ w_k_0_w + w_k_0_b
    V = inp_array @w_v_0_w + w_v_0_b

    dk = w_k_0_w.shape[0]

    A = Q @ K.transpose() / math.sqrt(dk)

    seq_len = len(words)
    mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)

    A_w = np.where(mask, float('-inf'), A)

    O = A_w @ V

    # pass it to ff nn
    hidden = O @ ff_line_1_w.transpose() + ff_line_1_b
    activated = gelu(hidden)
    ff_output = hidden @ ff_line_2_w.transpose() + ff_line_2_b

    # generate prediction using lm_head
    prediction_matrix = ff_output @ lm_head_w.transpose()
    prediction = softmax(prediction_matrix)[-1]

    next_word = vocab[prediction.tolist().index(max(prediction))]

    if next_word != 'END':
        return process([*words, next_word])
    return words

In [51]:
res = process(inp_words)
print(' '.join(res))

the cat sat red big big red red PAD big red


/var/folders/t0/kh3v7g4x5xvct_h780gjw8pm0000gn/T/ipykernel_946/156101372.py:24: RuntimeWarning: invalid value encountered in matmul
  O = A_w @ V
/var/folders/t0/kh3v7g4x5xvct_h780gjw8pm0000gn/T/ipykernel_946/156101372.py:27: RuntimeWarning: invalid value encountered in matmul
  hidden = O @ ff_line_1_w.transpose() + ff_line_1_b
